# 05 · Optimizer Comparison

- **Objectives**: train the same tiny model with all four optimizers (AdamW, Muon, Lion, Schedule-Free AdamW) and plot loss curves side by side.
- **Requirements**: 1 GPU (strongly preferred — CPU is ~10× slower).
- **Runtime**: ~2 minutes (4 optimizers × 100 steps).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Training harness

One function, parameterized by `OptimizerConfig`. Returns per-step loss values.

Each optimizer has different LR conventions — set them per-optimizer below, not the same number.

In [ ]:
from kempnerforge.config.schema import ModelConfig, OptimizerConfig
from kempnerforge.model.transformer import Transformer
from kempnerforge.training.optimizer import build_optimizer

MODEL_CONFIG = ModelConfig(
    dim=128,
    n_layers=4,
    n_heads=4,
    n_kv_heads=4,
    vocab_size=256,
    max_seq_len=64,
)

BATCH, SEQ, STEPS = 4, 32, 100


def train_once(opt_config: OptimizerConfig, seed: int = 0) -> list[float]:
    """Fresh model + optimizer, run `STEPS` steps on the same synthetic task."""
    torch.manual_seed(seed)
    model = Transformer(MODEL_CONFIG).to(device)
    optimizer = build_optimizer(model, opt_config)

    # Re-seed so every optimizer sees identical batches
    torch.manual_seed(seed + 1)
    losses = []
    for _ in range(STEPS):
        tokens = torch.randint(0, MODEL_CONFIG.vocab_size, (BATCH, SEQ + 1), device=device)
        inp, tgt = tokens[:, :-1], tokens[:, 1:]
        logits = model(inp)
        loss = F.cross_entropy(logits.reshape(-1, MODEL_CONFIG.vocab_size), tgt.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

## Run all four optimizers

LR choices follow each optimizer's own convention — using the same LR across all of them would be unfair:
- **AdamW** — `3e-4`, standard for transformers.
- **Muon** — `0.02`, ~50–100× larger than AdamW. The Newton-Schulz orthogonalization normalizes update magnitudes, so Muon operates in a very different LR regime (Jordan 2024). 1D params (biases, norms) fall back to AdamW internally at the same base LR.
- **Lion** — `3e-5`, ~10× smaller than AdamW. Sign-based updates have unit magnitude regardless of gradient scale, so the effective step is much larger per unit of LR.
- **Schedule-Free AdamW** — `0.025`, ~80× larger than AdamW. Schedule-Free averages iterates on the fly, which dampens effective per-step motion — you compensate with a larger base LR (Defazio & Mishchenko 2024).

In [ ]:
configs = {
    "adamw": OptimizerConfig(name="adamw", lr=3e-4),
    "muon": OptimizerConfig(name="muon", lr=0.02),
    "lion": OptimizerConfig(name="lion", lr=3e-5),
    "schedule_free_adamw": OptimizerConfig(name="schedule_free_adamw", lr=0.025),
}

all_losses: dict[str, list[float]] = {}
for name, cfg in configs.items():
    print(f"Training with {name}...", end=" ", flush=True)
    all_losses[name] = train_once(cfg)
    print(f"final loss = {all_losses[name][-1]:.3f}")

## Plot loss curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, losses in all_losses.items():
    ax.plot(losses, label=name, linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title(f"Optimizer comparison — {STEPS} steps, tiny model, synthetic data")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Final losses

In [ ]:
for name, losses in sorted(all_losses.items(), key=lambda kv: kv[1][-1]):
    print(f"  {name:<22} final={losses[-1]:.3f}   avg last 10 = {sum(losses[-10:]) / 10:.3f}")

## Caveats

- **The task is unlearnable.** Next-token prediction on fresh random tokens has no structure to learn, so every curve hovers near `log(vocab_size) ≈ 5.54`. The plot shows optimizer *stability and noise characteristics*, not learning progress.
- **Single-run comparison on synthetic data** — not a benchmark. Real optimizer comparisons need multiple seeds, a real tokenized corpus, and a proper LR sweep per optimizer.
- **No scheduler** — all runs use constant LR. In practice, AdamW/Muon/Lion benefit from warmup + cosine decay; Schedule-Free is designed to work without one.
- **Too few steps** for Muon's orthogonalization to stand out — Muon typically helps most on mid-training stability, not initial descent.

To get meaningful comparisons, swap the synthetic data for a real tokenized corpus and train for at least a few thousand steps.